# 🚀 Matrix Synapse Server on Google Colab

This notebook sets up a complete Matrix Synapse chat server directly in Google Colab!

## Features:
- ✅ **No sudo required** - User-level installation
- ✅ **SQLite database** - No PostgreSQL complexity
- ✅ **Element Web client** - Ready-to-use web interface
- ✅ **Google Drive persistence** - Save your data between sessions
- ✅ **HTTP tunneling** - Access from anywhere
- ✅ **Simple setup** - Just run the cells!

## 📱 What is Matrix?
Matrix is an open source chat protocol like Discord or Slack, but:
- **Decentralized** - You own your data
- **Federated** - Connect with other Matrix servers
- **Encrypted** - End-to-end encryption built-in
- **Open** - Use any Matrix client app

---

## 📥 Step 1: Download and Install Matrix Synapse

Run this cell to download and install Matrix Synapse with all dependencies:

In [ ]:
# Download and run Matrix Synapse installer for Colab
!curl -fsSL https://raw.githubusercontent.com/pablety/matrix-raspberry-installer/main/install-matrix-colab.sh -o install-matrix-colab.sh
!chmod +x install-matrix-colab.sh
!./install-matrix-colab.sh

## ☁️ Step 2: Setup Google Drive Persistence (Optional)

If you want to save your Matrix data between Colab sessions, run this cell:

In [ ]:
# Mount Google Drive for data persistence
from google.colab import drive
import os

print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Check if backup exists
backup_path = '/content/drive/MyDrive/Matrix_Backup/matrix_colab_backup.zip'
if os.path.exists(backup_path):
    print("📥 Found existing Matrix backup! Restoring...")
    !cd /content/matrix-colab && python sync-gdrive.py restore
    print("✅ Backup restored successfully")
else:
    print("ℹ️  No existing backup found. Your data will be backed up automatically.")

print("🎉 Google Drive setup complete!")

## 🚀 Step 3: Start Matrix Server

Run this cell to start your Matrix server:

In [ ]:
import subprocess
import time
import requests
from IPython.display import display, HTML

# Start Matrix server in background
print("🚀 Starting Matrix Synapse server...")
process = subprocess.Popen(['/content/matrix-colab/start-matrix.sh'], 
                          stdout=subprocess.PIPE, 
                          stderr=subprocess.STDOUT, 
                          text=True)

# Wait for server to start
time.sleep(10)

# Check if server is running
try:
    response = requests.get('http://localhost:8008/_matrix/client/versions', timeout=5)
    if response.status_code == 200:
        print("✅ Matrix server is running!")
        
        # Display access information
        display(HTML("""
        <div style="background: #e8f5e8; padding: 20px; border-radius: 10px; margin: 10px 0;">
            <h3>🎉 Matrix Server is Live!</h3>
            <p><strong>Matrix API:</strong> <a href="http://localhost:8008" target="_blank">http://localhost:8008</a></p>
            <p><strong>Element Web:</strong> <a href="http://localhost:8080" target="_blank">http://localhost:8080</a></p>
            <p><em>Note: Links will only work if you're running this locally. In Colab, use the tunneling in Step 4.</em></p>
        </div>
        """))
        
    else:
        print("❌ Server started but not responding correctly")
        print(f"Status: {response.status_code}")
        
except requests.exceptions.RequestException as e:
    print("❌ Server is not responding")
    print(f"Error: {e}")
    print("\n📋 Server output:")
    # Read some output
    for i in range(10):
        line = process.stdout.readline()
        if line:
            print(line.strip())
        else:
            break

print("\n📝 Next: Create a user account in Step 4!")

## 🌐 Step 4: Create Public Tunnel (Access from Internet)

To access your Matrix server from outside Colab, we'll create a public tunnel using ngrok:

In [ ]:
# Install and setup ngrok for public access
!pip install pyngrok

from pyngrok import ngrok
import time

# Create tunnels for Matrix API and Element Web
print("🌐 Creating public tunnels...")

# Tunnel for Matrix API (port 8008)
matrix_tunnel = ngrok.connect(8008, "http")
matrix_url = matrix_tunnel.public_url

# Tunnel for Element Web (port 8080)
element_tunnel = ngrok.connect(8080, "http")
element_url = element_tunnel.public_url

print(f"✅ Matrix API tunnel: {matrix_url}")
print(f"✅ Element Web tunnel: {element_url}")

# Display clickable links
from IPython.display import display, HTML

display(HTML(f"""
<div style="background: #e8f4f8; padding: 20px; border-radius: 10px; margin: 10px 0;">
    <h3>🌍 Your Matrix Server is Now Public!</h3>
    <p><strong>🔗 Matrix API:</strong> <a href="{matrix_url}" target="_blank">{matrix_url}</a></p>
    <p><strong>🎨 Element Web:</strong> <a href="{element_url}" target="_blank">{element_url}</a></p>
    <p><em>Share these URLs with friends to let them connect!</em></p>
    <p><strong>⚠️ Important:</strong> These URLs will change each time you restart the notebook.</p>
</div>
"""))

# Save URLs for later use
with open('/content/matrix-colab/tunnel_urls.txt', 'w') as f:
    f.write(f"Matrix API: {matrix_url}\n")
    f.write(f"Element Web: {element_url}\n")

print("\n📝 URLs saved to /content/matrix-colab/tunnel_urls.txt")
print("📱 Next: Create your user account in Step 5!")

## 👤 Step 5: Create Your First User Account

Run this cell to create an admin user account:

In [ ]:
import subprocess
import sys
from getpass import getpass

print("👤 Creating Matrix user account...")
print("📝 You'll need to enter:")
print("   - Username (e.g., 'admin', 'yourname')")
print("   - Password (secure password)")
print("   - Admin status (yes/no)")
print("")

# Alternative: Create user programmatically
print("🔧 Creating admin user programmatically...")

username = input("Enter username: ").strip()
password = getpass("Enter password: ").strip()
is_admin = input("Make admin? (y/n): ").strip().lower()

if username and password:
    admin_flag = "--admin" if is_admin in ['y', 'yes'] else "--no-admin"
    
    # Run user creation command
    cmd = [
        '/content/matrix-colab/env/bin/register_new_matrix_user',
        '-u', username,
        '-p', password,
        admin_flag,
        '-c', '/content/matrix-colab/homeserver.yaml',
        'http://localhost:8008'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            print(f"✅ User '{username}' created successfully!")
            
            # Show login info
            from IPython.display import display, HTML
            
            # Read tunnel URLs
            matrix_url = "http://localhost:8008"
            element_url = "http://localhost:8080"
            
            try:
                with open('/content/matrix-colab/tunnel_urls.txt', 'r') as f:
                    lines = f.readlines()
                    matrix_url = lines[0].split(': ')[1].strip()
                    element_url = lines[1].split(': ')[1].strip()
            except:
                pass
            
            display(HTML(f"""
            <div style="background: #e8f5e8; padding: 20px; border-radius: 10px; margin: 10px 0;">
                <h3>🎉 Account Created Successfully!</h3>
                <p><strong>Username:</strong> {username}</p>
                <p><strong>Server:</strong> colab-matrix.local</p>
                <p><strong>Login URL:</strong> <a href="{element_url}" target="_blank">{element_url}</a></p>
                <hr>
                <h4>📱 How to Login:</h4>
                <ol>
                    <li>Click the Element Web link above</li>
                    <li>Click "Sign In"</li>
                    <li>Enter your username: <code>{username}</code></li>
                    <li>Enter your password</li>
                    <li>For server, use: <code>{matrix_url}</code></li>
                </ol>
            </div>
            """))
            
        else:
            print(f"❌ Failed to create user: {result.stderr}")
            
    except subprocess.TimeoutExpired:
        print("❌ User creation timed out")
    except Exception as e:
        print(f"❌ Error creating user: {e}")
        
else:
    print("❌ Username and password are required")

print("\n🎊 Your Matrix server is ready to use!")

## 💾 Step 6: Backup Your Data

Before your Colab session ends, backup your Matrix data to Google Drive:

In [ ]:
import os
import subprocess

print("💾 Backing up Matrix data to Google Drive...")

# Check if Google Drive is mounted
if not os.path.exists('/content/drive'):
    print("📁 Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

# Run backup
try:
    result = subprocess.run(
        ['python', '/content/matrix-colab/sync-gdrive.py', 'backup'],
        capture_output=True,
        text=True,
        timeout=120
    )
    
    if result.returncode == 0:
        print("✅ Backup completed successfully!")
        print(result.stdout)
        
        # Show backup info
        backup_path = '/content/drive/MyDrive/Matrix_Backup/matrix_colab_backup.zip'
        if os.path.exists(backup_path):
            size_mb = os.path.getsize(backup_path) / 1024 / 1024
            print(f"📊 Backup size: {size_mb:.1f} MB")
            
            from IPython.display import display, HTML
            display(HTML("""
            <div style="background: #fff3cd; padding: 15px; border-radius: 8px; margin: 10px 0;">
                <h4>📂 Backup Location</h4>
                <p>Your Matrix data is saved in:</p>
                <code>Google Drive → MyDrive → Matrix_Backup → matrix_colab_backup.zip</code>
                <p><strong>💡 Tip:</strong> This backup will be automatically restored next time you run this notebook!</p>
            </div>
            """))
        
    else:
        print("❌ Backup failed:")
        print(result.stderr)
        
except subprocess.TimeoutExpired:
    print("❌ Backup timed out")
except Exception as e:
    print(f"❌ Backup error: {e}")

print("\n🎉 Backup process complete!")

## 📊 Server Monitoring & Management

Use these cells to monitor and manage your Matrix server:

In [ ]:
# Check server status
import requests
import json
import os
from IPython.display import display, HTML

def check_server_status():
    try:
        # Check Matrix API
        response = requests.get('http://localhost:8008/_matrix/client/versions', timeout=5)
        matrix_status = "✅ Running" if response.status_code == 200 else "❌ Error"
    except:
        matrix_status = "❌ Not running"
    
    try:
        # Check Element Web
        response = requests.get('http://localhost:8080', timeout=5)
        element_status = "✅ Running" if response.status_code == 200 else "❌ Error"
    except:
        element_status = "❌ Not running"
    
    # Check database size
    db_size = "Unknown"
    db_path = "/content/matrix-colab/data/synapse.db"
    if os.path.exists(db_path):
        size_bytes = os.path.getsize(db_path)
        db_size = f"{size_bytes / 1024 / 1024:.1f} MB"
    
    # Check tunnel URLs
    tunnel_info = "Not configured"
    try:
        with open('/content/matrix-colab/tunnel_urls.txt', 'r') as f:
            tunnel_info = f.read().strip()
    except:
        pass
    
    display(HTML(f"""
    <div style="background: #f8f9fa; padding: 20px; border-radius: 10px; font-family: monospace;">
        <h3>📊 Matrix Server Status</h3>
        <table style="width: 100%; border-collapse: collapse;">
            <tr><td><strong>Matrix API:</strong></td><td>{matrix_status}</td></tr>
            <tr><td><strong>Element Web:</strong></td><td>{element_status}</td></tr>
            <tr><td><strong>Database Size:</strong></td><td>{db_size}</td></tr>
            <tr><td colspan="2"><hr></td></tr>
            <tr><td colspan="2"><strong>Public URLs:</strong><br><pre>{tunnel_info}</pre></td></tr>
        </table>
    </div>
    """))

check_server_status()

In [ ]:
# View server logs
import os

log_file = "/content/matrix-colab/synapse.log"

if os.path.exists(log_file):
    print("📋 Latest Matrix server logs:")
    print("=" * 50)
    
    # Show last 20 lines
    !tail -n 20 /content/matrix-colab/synapse.log
else:
    print("📋 No log file found. Server may not be running.")

print("\n💡 To view live logs, use: !tail -f /content/matrix-colab/synapse.log")

In [ ]:
# Restart Matrix server
import subprocess
import time

print("🔄 Restarting Matrix server...")

# Stop existing processes
!pkill -f synapse
!pkill -f "http.server 8080"

time.sleep(3)
print("🛑 Stopped existing processes")

# Start server again
print("🚀 Starting server...")
process = subprocess.Popen(['/content/matrix-colab/start-matrix.sh'], 
                          stdout=subprocess.PIPE, 
                          stderr=subprocess.STDOUT, 
                          text=True)

time.sleep(8)

# Check if restarted successfully
try:
    import requests
    response = requests.get('http://localhost:8008/_matrix/client/versions', timeout=5)
    if response.status_code == 200:
        print("✅ Server restarted successfully!")
    else:
        print("❌ Server restart failed")
except:
    print("❌ Server not responding after restart")

print("\n📝 Check the status cell above to verify all services are running")

## 💡 Usage Tips & Troubleshooting

### 🔧 Common Commands
- **Check status:** Run the "Server Status" cell above
- **View logs:** Run the "View Logs" cell above  
- **Restart server:** Run the "Restart Server" cell above
- **Backup data:** Run Step 6 before ending your session

### 📱 Connecting Matrix Clients
You can use any Matrix client with your server:

**Desktop/Web:**
- Element Web (built-in)
- Element Desktop
- SchildiChat
- Cinny

**Mobile:**
- Element Android/iOS
- SchildiChat Mobile
- FluffyChat

**Connection Settings:**
- **Server URL:** Use the Matrix API tunnel URL from Step 4
- **Username:** The username you created in Step 5
- **Password:** The password you set

### ⚠️ Important Notes
- **Session Limits:** Google Colab sessions have time limits (12-24 hours)
- **Data Persistence:** Always backup to Google Drive before ending sessions
- **URLs Change:** Tunnel URLs change each time you restart
- **Performance:** Colab has limited resources, suitable for testing/small groups

### 🐛 Troubleshooting

**Server won't start:**
1. Check the logs cell above
2. Try restarting (restart cell above)
3. If still failing, re-run the installation (Step 1)

**Can't access from outside:**
1. Make sure Step 4 (tunneling) was completed
2. Check if tunnel URLs are still active
3. Re-run Step 4 to create new tunnels

**Data lost:**
1. Check Google Drive backup in MyDrive/Matrix_Backup/
2. Re-run Step 2 to restore from backup
3. Make sure to backup regularly (Step 6)

---

## 🎉 Enjoy Your Matrix Server!

You now have a fully functional Matrix chat server running in Google Colab! 

**Share the Element Web URL with friends and start chatting!** 💬
